In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import random
from PIL import Image
import pandas as pd
import ast
import matplotlib.gridspec as gridspec

from DWT import DWT_2D
from IXCM import IXCNet

In [2]:
class TB11KDataset(Dataset):
    def __init__(self, list_path, img_root_path, transform=None):
        self.img_root_path = img_root_path
        self.transform = transform
        self.img_paths = []
        self.labels = []

        with open(list_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                rel_path = parts[0]
                full_path = os.path.join(self.img_root_path, rel_path)
                
                if '/tb/' in rel_path or rel_path.startswith('tb/'):
                    label = 1.0
                else:
                    label = 0.0
                
                self.img_paths.append(full_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert('L') 
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            image = Image.new('L', (256, 256))

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor([label], dtype=torch.float32)

In [3]:
def visualize_evidence_explainability(model, dwt_module, val_loader, class_names, device, sample_idx=0, batch_id=1, save_dir=''):
    model.eval()
    dwt_module.eval()

    target_batch_idx = batch_id 
    images = None
    labels = None
    
    for i, (imgs, lbls) in enumerate(val_loader):
        if i == target_batch_idx:
            images = imgs
            labels = lbls
            break 
    
    if images is None:
        print("Error: Batch ID out of range.")
        return

    images = images.to(device)
    b_idx = sample_idx
    
    with torch.no_grad():
        s_tensor, lh, hl, hh = dwt_module(images)
        a_tensor = lh + hl
        alpha_s_list = []
        alpha_a_list = []
        beta_s_list = []
        beta_a_list = []
        for cls_name in class_names:
            curr_model = model
            output = curr_model(images)
            
            alphas = output[0][b_idx].cpu().numpy()
            betas  = output[1][b_idx].cpu().numpy()
            
            alpha_s_list.append(alphas[0])
            alpha_a_list.append(alphas[1])
            beta_s_list.append(betas[0])
            beta_a_list.append(betas[1])

    alpha_s_vals = np.array(alpha_s_list)
    beta_s_vals  = np.array(beta_s_list)
    alpha_a_vals = np.array(alpha_a_list)
    beta_a_vals  = np.array(beta_a_list)

    prob_s = alpha_s_vals / (alpha_s_vals + beta_s_vals + 1e-6)
    prob_a = alpha_a_vals / (alpha_a_vals + beta_a_vals + 1e-6)

    def to_img(t):
        tensor_img = t[b_idx, 0, :, :]  
        img = tensor_img.cpu().numpy()
        img_min = img.min()
        img_max = img.max()
        if img_max - img_min > 1e-6:
            img = (img - img_min) / (img_max - img_min)
        return img

    img_orig = to_img(images)
    img_s = to_img(s_tensor)
    img_a = to_img(a_tensor)
    
    curr_labels = labels[b_idx].numpy()
    num_classes = len(class_names)

    fig = plt.figure(figsize=(18, 12))

    outer_gs = gridspec.GridSpec(2, 1, height_ratios=[1.5, 0.8], hspace=0.01)

    top_gs = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=outer_gs[0], wspace=0.2)
    
    ax_orig = fig.add_subplot(top_gs[0])
    ax_s = fig.add_subplot(top_gs[1])
    ax_a = fig.add_subplot(top_gs[2])

    ax_orig.imshow(img_orig, cmap='gray')
    gt_classes = [class_names[i] for i in range(num_classes) if curr_labels[i]==1]
    gt_str = ",".join(gt_classes) if gt_classes else "None"
    ax_orig.set_title(f"Original CXR Image\nDisease(s): \n[{gt_str}]", fontsize=16, fontweight='bold')
    ax_orig.axis('off')
    
    ax_s.imshow(img_s, cmap='gray')
    ax_s.set_title("Structure (S)", fontsize=30, fontweight='bold', color='#1f77b4')
    ax_s.axis('off')
    
    ax_a.imshow(img_a, cmap='gray')
    ax_a.set_title("Texture (T)", fontsize=30, fontweight='bold', color='#d62728')
    ax_a.axis('off')

    bottom_gs = gridspec.GridSpecFromSubplotSpec(1, 3, 
                                                 subplot_spec=outer_gs[1], 
                                                 width_ratios=[1, 0.25, 1]) 
    
    ax_bar_s = fig.add_subplot(bottom_gs[0])
    ax_bar_a = fig.add_subplot(bottom_gs[2])

    def plot_bar(ax, alpha_vals, beta_vals, probs, color, title):
        y_pos = np.arange(num_classes)
        bars = ax.barh(y_pos, probs, align='center', color=color, alpha=0.85, edgecolor='white')
        
        ax.set_yticks(y_pos)

        ax.set_yticklabels(class_names, fontsize=20, fontweight='medium')
        ax.invert_yaxis()
        if title.split(' ')[0] == 'Structure':
            ax.set_xlabel('Structural Positive Probability', fontsize=20)
        else:
            ax.set_xlabel('Textural Positive Probability', fontsize=20)
        ax.set_xlim(0, 1.0)
        ax.tick_params(axis='x', labelsize=15)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        ax.set_title(title, fontsize=26, fontweight='bold', pad=15)
        ax.grid(axis='x', linestyle='--', alpha=0.4)
        
        for i, bar in enumerate(bars):
            is_gt = curr_labels[i] == 1
            font_w = 'bold' if is_gt else 'normal'
            txt_color = 'black' if is_gt else '#555555'
            label_text = f" {probs[i]:.2f} (α={alpha_vals[i]:.2f},β={beta_vals[i]:.2f})"

            bar_width = bar.get_width()
            if bar_width > 0.6:
                ax.text(bar_width - 0.02, bar.get_y() + bar.get_height()/2, 
                        label_text, 
                        va='center', ha='right', 
                        fontsize=18, fontweight=font_w, color=txt_color)
            else:
                ax.text(bar_width + 0.01, bar.get_y() + bar.get_height()/2, 
                        label_text, 
                        va='center', ha='left', 
                        fontsize=18, fontweight=font_w, color=txt_color)

    plot_bar(ax_bar_s, alpha_s_vals, beta_s_vals, prob_s, '#89CFF0', "Structural Evidence")

    plot_bar(ax_bar_a, alpha_a_vals, beta_a_vals, prob_a, '#F4C2C2', "Textural Evidence")
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(os.path.join(save_dir, f'{sample_idx}_{batch_id}.png'), dpi=300)
    plt.show()

In [4]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Random Seed set to: {seed}")

In [5]:
seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 64
IMG_ROOT = ''
TRAIN_LIST = ''
VAL_LIST = ''

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]) 
    ]),
    'val': transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ]),
}

train_dataset = TB11KDataset(TRAIN_LIST, IMG_ROOT, transform=data_transforms['train'])
val_dataset = TB11KDataset(VAL_LIST, IMG_ROOT, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

Random Seed set to: 42
Train samples: 6600, Val samples: 1800


In [ ]:
DEVICE = 'cuda'
save_dir = ''
model_root_path = ''
final_model = IXCNet(evi_vec_dim=64, wavename='haar', ydim=1, in_channels=1, img_dec=True).to(DEVICE)
final_model.to(DEVICE)
best_model_path = os.path.join(model_root_path, 'best_model_auc.pth')
if os.path.exists(best_model_path):
    print(f"Loading weights from: {best_model_path}")
    final_model.load_state_dict(torch.load(best_model_path))
else:
    print("Warning: Best model not found, using current weights.")

wavename = 'haar'
dwt2D = DWT_2D(wavename).to(DEVICE)
my_classes = ['TB'] 
batch_id = 1

In [ ]:
visualize_evidence_explainability(final_model, dwt2D, val_loader, my_classes, DEVICE, sample_idx=5, batch_id=batch_id, save_dir=save_dir)